In [ ]:
# import all packages
import numpy as np
import pandas as pd
import glob, os
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import math
import statsmodels.formula.api as sm
from statsmodels.stats.anova import anova_lm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Read roadway data

In [ ]:
"""### Read roadway lengths"""
import os

BASE_DIR = "/content/drive/MyDrive/quantify-infrastructure-code"
folder_path = os.path.join(BASE_DIR, "data/osm/All states")

# Get a list of all CSV files in the folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

# Initialize an empty DataFrame to store concatenated data
concatenated_df = pd.DataFrame()

# Loop through each CSV file and concatenate them
for file in csv_files:
    # print(file)
    file_path = os.path.join(folder_path, file)

    # Read the CSV file into a DataFrame
    df_state = pd.read_csv(file_path, index_col =0)
    # to check how many places are excluded
    # print(df_state.shape[0])
    # count = count + df_state.shape[0]
    df_state['GEOID'] = df_state['GEOID'].astype(str).str.rjust(7, '0')

    # Concatenate the DataFrame to the existing data
    concatenated_df = pd.concat([concatenated_df, df_state], ignore_index=True)
print(concatenated_df.shape)

# 13 missing places due to missing geometry
# Total no of places for 50 states in 2020
# 31249+13 = 31262

concatenated_df[['GEOID', 'NAMELSAD',  'secondary', 'tertiary', 'unclassified', 'residential', 'cl_tertiary', 'cl_unclassified', 'cl_residential', 'cl_service',
                'lane_m_tertiary', 'lane_m_unclassified', 'lane_m_residential', 'lane_m_service', 'lane_m_living_street']]

df = concatenated_df.copy()

# Calculate total roadway length and total centerline roadway length
roadway_columns = ['motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'unclassified', 'residential', 'service', 'living_street']
cl_roadway_columns = ['cl_motorway', 'cl_trunk', 'cl_primary', 'cl_secondary', 'cl_tertiary', 'cl_unclassified', 'cl_residential']
lane_m_roadway_columns = ['lane_m_motorway', 'lane_m_trunk', 'lane_m_primary','lane_m_secondary', 'lane_m_tertiary', 'lane_m_unclassified',
                        'lane_m_residential', 'lane_m_service', 'lane_m_track','lane_m_footway', 'lane_m_cycleway', 'lane_m_living_street']
cl_local_columns = ['cl_unclassified', 'cl_residential', 'cl_service'] # 'cl_secondary'
lane_m_local_columns =['lane_m_unclassified', 'lane_m_residential', 'lane_m_service','lane_m_living_street'] # 'lane_m_secondary', 'lane_m_track','lane_m_footway', 'lane_m_cycleway',
footway_columns = ['footway','n_residential','n_footway']

df['total_length'] = df[roadway_columns].sum(axis=1)
df['cl_all_classes'] = df[cl_roadway_columns].sum(axis=1)
df['lane_m_all_classes'] = df[lane_m_roadway_columns].sum(axis=1)
df['cl_local_length'] = df[cl_local_columns].sum(axis=1)
df['cl_total_length_2020'] = df[lane_m_local_columns].sum(axis=1)
df['walkway_length'] = df[footway_columns].sum(axis=1)

# df['road_area_m2'] = df['motorway'] * 13.6 + df['trunk'] * 9.6 + df['primary'] * 6.0 + df['secondary'] * 5.3 + df['tertiary'] * 4.9 +\
#       df['unclassified'] * 4.5 + df['residential'] * 4.5
df['road_lanearea_m2'] = df['lane_m_all_classes'] * 3.6

df['pct_cl_local'] = df['cl_local_length'] * 100 / df['cl_all_classes']
df['pct_local'] = df['cl_total_length_2020'] * 100 / df['lane_m_all_classes']
df['pct_walkway'] = df['walkway_length']*100/df['cl_total_length_2020'] # This value need to be checked since residential (auto) can have walkways too

print(f"Total cities with zero local roadways:==== {df[(df['cl_total_length_2020']==0)].shape[0]}")
print(f"CDPs with zero local roadways:==== {df[(df['NAMELSAD'].str.contains('CDP')) & (df['cl_total_length_2020']==0)].shape[0]}")
print(f"CDPs with local roadways:==== {df[(df['NAMELSAD'].str.contains('CDP')) & (df['cl_total_length_2020']!=0)].shape[0]}")
print(f"Total non-zero local lane-meter available for {df[(df['cl_total_length_2020']!=0)].shape[0]} cities")
# df[(df['NAMELSAD'].str.contains('CDP')) &(df['cl_total_length_2020']!=0)]

df[(df['NAMELSAD'].str.contains('CDP')) &(df['cl_total_length_2020']!=0)][['GEOID', 'NAMELSAD', 'total_length', 'cl_all_classes', 'lane_m_all_classes', 'cl_total_length_2020','pct_local']].sort_values(by=['pct_local']).round(2)

print(f"Total centerline length of local roadways: {df['cl_local_length'].sum().round(2)}")
print(f"Total lane meter length of local roadways: {df['cl_total_length_2020'].sum().round(2)}")

# NaNs values in roadway length dataframe
df.isna().sum().sum()

"""### Import population and attributes data from depopulation study"""

df_population = pd.read_csv('/content/drive/MyDrive/quantify-infrastructure-code/population/forecasted_trend.csv', index_col  = 0)
df_population['GEOID'] = df_population['GEOID'].astype(str).str.rjust(7,'0')

df_attributes = pd.read_csv('/content/drive/MyDrive/quantify-infrastructure-code/population/df_attributes.csv', index_col  = 0)
df_attributes['GEOID'] = df_attributes['GEOID'].astype(str).str.rjust(7,'0')

df_pop_attr = df_population.merge(df_attributes[['GEOID', 'REGION', 'city type', 'weighted_HU_density_sqmi','median_income']], on = 'GEOID')
print(df_population.shape, df_attributes.shape, df_pop_attr.shape)

# Total population for SSP 2 for each 10 yr interval
print('Total population in millions for 51 states: ===')
df_population[['CensusPop_20','ssp22020', 'ssp22030', 'ssp22040', 'ssp22050', 'ssp22060','ssp22070', 'ssp22080', 'ssp22090', 'ssp22100']].sum() / 1000000

"""### Merge roads with population"""

roads_with_pop_all = df.merge(df_pop_attr[['GEOID', 'State', 'REGION', 'ALAND', 'label', 'future trend from SSP 2', 'CensusPop_20', 'city type','weighted_HU_density_sqmi','median_income',
                                    'ssp22020', 'ssp22030', 'ssp22040', 'ssp22050','ssp22060', 'ssp22070', 'ssp22080',
                                    'ssp22090','ssp22100', 'ssp12020', 'ssp12030', 'ssp12040', 'ssp12050', 'ssp12060','ssp12070',
                                                'ssp12080', 'ssp12090', 'ssp12100', 'ssp42020', 'ssp42030', 'ssp42040', 'ssp42050', 'ssp42060',
                                                'ssp42070', 'ssp42080', 'ssp42090', 'ssp42100']], on = 'GEOID', how='left')
print(roads_with_pop_all.shape)

print(roads_with_pop_all[roads_with_pop_all['GEOID'].str.startswith('214800')][['GEOID', 'NAMELSAD','CensusPop_20', 'ssp22020', 'ssp22030', 'ssp22040', 'ssp22050']])
# 2010 Census
# Louisville/Jefferson County metro government (balance), Kentucky	597337
# 2020 Census
# Louisville city, Kentucky	246161
# Louisville/Jefferson County metro government (balance), Kentucky	386884

# 246161+386884 = 633045

print('Total population in millions for 50 states: ===')
print(roads_with_pop_all[['CensusPop_20', 'ssp22020', 'ssp22030', 'ssp22040', 'ssp22050', 'ssp22060','ssp22070', 'ssp22080', 'ssp22090', 'ssp22100']].sum() / 1000000)

roads_with_pop_all['percentchangeinpop'] = np.abs((roads_with_pop_all['ssp42020'] - roads_with_pop_all['CensusPop_20'])/roads_with_pop_all['CensusPop_20'])
# cities that have their 2020 census population with 1% variation of the projected ssp22020 population are included in the analysis
roads_with_pop = roads_with_pop_all[(roads_with_pop_all['percentchangeinpop'] <=.01)]
print(roads_with_pop.shape)
# roads_with_pop_all[roads_with_pop_all['percentchangeinpop'] > 0.01][['GEOID','NAMELSAD', 'CensusPop_20', 'ssp22020',]].sort_values(by='ssp22020')

roads_with_pop['road_density_m-sqm'] = roads_with_pop['cl_total_length_2020'] /roads_with_pop['ALAND']
roads_with_pop['road_density'] = roads_with_pop['road_lanearea_m2'] * 100/roads_with_pop['ALAND']

# Exclude total zero roadways
print(f"Total cities with nonzero local roadway lane meter {roads_with_pop[roads_with_pop['cl_total_length_2020']!=0].shape[0]}")

"""### Model comparision Pruned Nonpruned F_test"""

roads_with_pop['per_cap_mass_at_2020'] =  roads_with_pop['cl_total_length_2020'] / roads_with_pop['CensusPop_20']

stocks_with_pop = roads_with_pop[roads_with_pop['CensusPop_20'] != 0]
stocks_with_pop = roads_with_pop[roads_with_pop['per_cap_mass_at_2020'] != 0]
# stocks_with_pop = stocks_with_pop[stocks_with_pop['city type'] == 'suburban']
print(stocks_with_pop.shape)

pd.set_option('mode.chained_assignment', None) # To stop SettingWithCopy warning
list_of_city_types = ['urban', 'suburban', 'periurban', 'rural']

print('Total NaNs in population and roadway length:===')
print(roads_with_pop[['CensusPop_20', 'ssp22040', 'cl_total_length_2020']].isna().sum())
print("Places with no roadway:===", roads_with_pop[roads_with_pop['cl_total_length_2020'] == 0].shape[0])
print("Places with zero population in census 2020:===", roads_with_pop[roads_with_pop['CensusPop_20'] == 0].shape[0])
print("Places with no available population forecast:===", roads_with_pop[roads_with_pop['ssp22040'].isnull()].shape[0])
print(roads_with_pop.shape)

roads_clean = roads_with_pop.dropna(subset=['CensusPop_20', 'ssp22040', 'total_length', 'cl_total_length_2020']).reset_index(drop=True)
print(roads_clean.shape)
roads_clean = roads_clean[roads_clean['cl_total_length_2020']!=0]
print(roads_clean.shape)
current_roadway_column = 'cl_total_length_2020' # 'road_area_m2' #

roads_clean['per_cap_mass_at_2020'] = roads_clean[current_roadway_column] / roads_clean['CensusPop_20'].round(0)
print("Shape of the clean dataset with nonzero values:==")
print(roads_clean.shape)
print(roads_clean[(roads_clean['per_cap_mass_at_2020'] > 10000) | (roads_clean['total_length'] < 500)][['GEOID','NAMELSAD','CensusPop_20','total_length', 'cl_total_length_2020','per_cap_mass_at_2020']])
# roads_clean = roads_clean.merge(df_urban_rural_conn, on ='GEOID')
roads_clean = roads_clean[(roads_clean['per_cap_mass_at_2020'] <= 10000) & (roads_clean['total_length'] >= 500)]
print("Shape of the clean dataset with newly defined urban rural continuum at each time interval values:==")
print(roads_clean.shape)

roads_clean.groupby(['city type','REGION'])[['pct_local', 'pct_cl_local','per_cap_mass_at_2020']].median().sort_values(['city type','pct_cl_local']).round(2)

roads_with_pop.shape, df.shape

roads_clean['city_type_order'] = roads_clean['city type'].map({'urban': 1, 'suburban': 2, 'periurban': 3, 'rural': 4, 'not enough data': 5})
roads_clean['REGION_order'] = roads_clean['REGION'].map({'Northeast': 1, 'Midwest': 2, 'West': 3, 'South': 4})

roads_clean[roads_clean['per_cap_mass_at_2020']>1000][['GEOID', 'NAMELSAD', 'city type', 'REGION', 'ALAND', 'CensusPop_20','per_cap_mass_at_2020']].sort_values('per_cap_mass_at_2020').shape

roads_clean.groupby('REGION')[['lane_m_motorway', 'lane_m_trunk', 'lane_m_primary', 'lane_m_secondary', 'lane_m_tertiary',
    'lane_m_unclassified', 'lane_m_residential', 'lane_m_service',
    'lane_m_living_street']].sum().sum(axis=1)

roads_clean['pct_local_lane-m'] = roads_clean['cl_total_length_2020'] / roads_clean[['lane_m_motorway', 'lane_m_trunk', 'lane_m_primary', 'lane_m_secondary', 'lane_m_tertiary',
    'lane_m_unclassified', 'lane_m_residential', 'lane_m_service', 'lane_m_living_street']].sum(axis=1)

roads_clean.groupby('REGION')['pct_local_lane-m'].describe()

roads_clean[~roads_clean['NAMELSAD'].str.contains('CDP')].sort_values('pct_local_lane-m')[['GEOID', 'NAMELSAD','CensusPop_20','cl_total_length_2020','pct_local_lane-m', 'REGION', 'city type']].head(12)

(31257, 74)
Total cities with zero local roadways:==== 61
CDPs with zero local roadways:==== 50
CDPs with local roadways:==== 11837
Total non-zero local lane-meter available for 31196 cities
Total centerline length of local roadways: 1624750936.58
Total lane meter length of local roadways: 3254473174.95
(31617, 58) (31908, 33) (31616, 62)
Total population in millions for 51 states: ===
(31257, 120)
         GEOID                                           NAMELSAD  \
10248  2148006  Louisville/Jefferson County metro government (...   
10408  2148000                                    Louisville city   

       CensusPop_20     ssp22020     ssp22030     ssp22040     ssp22050  
10248      386884.0  389068.4372  424417.2341  456660.9352  485264.1312  
10408           NaN          NaN          NaN          NaN          NaN  
Total population in millions for 50 states: ===
CensusPop_20    248.780912
ssp22020        250.288598
ssp22030        270.512914
ssp22040        288.565991
ssp22050    

C:\Users\User\AppData\Local\Temp\ipykernel_9720\784340813.py:116: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  roads_with_pop['road_density_m-sqm'] = roads_with_pop['cl_total_length_2020'] /roads_with_pop['ALAND']
C:\Users\User\AppData\Local\Temp\ipykernel_9720\784340813.py:117: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  roads_with_pop['road_density'] = roads_with_pop['road_lanearea_m2'] * 100/roads_with_pop['ALAND']
C:\Users\User\AppData\Local\Temp\ipykernel_9720\784340813.py:124: SettingWithCopyWarnin

(30578, 124)
(30537, 124)
Shape of the clean dataset with nonzero values:==
(30537, 124)
         GEOID              NAMELSAD  CensusPop_20   total_length  \
686    0401170        Alamo Lake CDP           4.0  120493.451000   
4057   1205150   Belleair Shore town          73.0     172.774243   
15235  3063266  Riverview Colony CDP         100.0      15.341483   
17663  3664881      Saltaire village         113.0       7.194151   

       cl_total_length_2020  per_cap_mass_at_2020  
686           100278.693928          25069.673482  
4057             149.044847              2.041710  
15235             15.341483              0.153415  
17663              7.194151              0.063665  
Shape of the clean dataset with newly defined urban rural continuum at each time interval values:==
(30533, 124)


,GEOID,NAMELSAD,CensusPop_20,cl_total_length_2020,pct_local_lane-m,REGION,city type
22502,4210240,Burlington borough,144.0,115.702876,0.015691,Northeast,rural
4093,1275625,Weeki Wachee city,16.0,270.441690,0.016444,South,periurban
12852,2742092,Miesville city,138.0,431.742629,0.043990,Midwest,rural
19871,3988070,Zanesfield village,194.0,232.063708,0.050427,Midwest,rural
5082,1351968,Mitchell town,153.0,914.583874,0.057618,South,rural
3921,1033250,Hartly town,73.0,188.727150,0.061030,South,rural
10485,2252565,Mound village,12.0,1477.792082,0.068294,South,rural
14191,2978928,West Plains city,12184.0,15752.210042,0.070517,Midwest,suburban
17677,3629872,Grand View-on-Hudson village,246.0,543.862765,0.072316,Northeast,rural
20613,3931542,Graysville village,70.0,814.747840,0.087029,Midwest,rural


In [ ]:
roads_with_pop_all = roads_clean # [roads_clean['State'] !=15].reset_index(drop=True)

print("Shape of building dataframe", roads_with_pop_all.shape[0])

Shape of building dataframe 30533


In [ ]:
roads_with_pop_all['State'].nunique()

50

In [ ]:
roads_with_pop_all[['GEOID', 'city type', 'CensusPop_20', 'ALAND', 'road_density', 'road_density_m-sqm']].sort_values('CensusPop_20')

,GEOID,city type,CensusPop_20,ALAND,road_density,road_density_m-sqm
21956,4131050,rural,3.0,2.227600e+05,3.388469,0.009124
14343,2962056,rural,3.0,3.937865e+06,2.577474,0.001909
24798,4666660,rural,3.0,6.517780e+05,3.255069,0.004047
27606,4933027,rural,4.0,3.482130e+07,0.320815,0.000091
15291,3080275,rural,4.0,1.130160e+05,4.957959,0.003433
...,...,...,...,...,...,...
668,0455000,suburban,1608139.0,1.341602e+09,6.279303,0.010206
25979,4835000,urban,2304580.0,1.658743e+09,7.981720,0.009276
7189,1714000,urban,2746388.0,5.898185e+08,8.723567,0.011876
1914,0644000,urban,3898747.0,1.215979e+09,9.254716,0.013141


In [ ]:
# roads_with_pop_all['State'].value_counts()

### ROADWAY FORECASTING FRAMEWORK

In [ ]:
# =============================================================================
# ROADWAY FORECASTING FRAMEWORK
#
# State Fixed Effects
# + Growth Factor Forecasting
# + Path Dependence
# + Hysteresis for Shrinking Cities
#
# =============================================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# =============================================================================
# INPUT DATA
# =============================================================================

df_ex_DC = roads_with_pop_all[roads_with_pop_all['State']!=11]


df_model = df_ex_DC[
    [
        'GEOID',
        'State',
        'REGION',
        'city type',
        'ALAND',
        'CensusPop_20',
        'ssp22030',
        'ssp22040',
        'ssp22050',
        'ssp22060',
        'ssp22070',
        'ssp22080',
        'ssp22090',
        'ssp22100',
        'cl_total_length_2020'
    ]
].copy()

df_model = df_model.rename(
    columns={
        'city type':'city_type'
    }
)

# =============================================================================
# REMOVE INVALID VALUES
# =============================================================================

df_model = df_model[
    (df_model['CensusPop_20'] > 0) &
    (df_model['ALAND'] > 0) &
    (df_model['cl_total_length_2020'] > 0)
].copy()

# =============================================================================
# LOG TRANSFORMS
# =============================================================================

df_model['log_pop'] = np.log(
    df_model['CensusPop_20']
)

df_model['log_area'] = np.log(
    df_model['ALAND']
)

df_model['log_rl'] = np.log(
    df_model['cl_total_length_2020']
)

# =============================================================================
# TRAIN / TEST SPLIT
# =============================================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

train_df, test_df = train_test_split(
    df_model,
    test_size=0.20,
    random_state=42,
    stratify=df_model['State']
)

# =============================================================================
# FIT STATE FIXED EFFECTS MODEL
# =============================================================================

model = smf.ols(
    formula=
    "log_rl ~ log_pop + log_area + C(State)",
    data=train_df
).fit()

print(model.summary())

# =============================================================================
# TEST PERFORMANCE
# =============================================================================

test_df = test_df.copy()

test_df["pred_log_rl"] = model.predict(
    test_df
)

test_df["pred_rl"] = np.exp(
    test_df["pred_log_rl"]
)

r2 = r2_score(
    test_df["cl_total_length_2020"],
    test_df["pred_rl"]
)

print(f"\nTest R² = {r2:.4f}")

# =============================================================================
# REFIT USING FULL DATASET
# =============================================================================

final_model = smf.ols(
    formula=
    "log_rl ~ log_pop + log_area + C(State)",
    data=df_model
).fit()

# =============================================================================
# BASELINE MODEL PREDICTION (2020)
# =============================================================================

df_model["Model_RL_2020"] = np.exp(
    final_model.predict(df_model)
)

# =============================================================================
# OBSERVED 2020 NETWORK
# =============================================================================

df_model["Predicted_RL_2020"] = (
    df_model["cl_total_length_2020"]
)

# =============================================================================
# FORECAST YEARS
# =============================================================================

years = [
    "2030",
    "2040",
    "2050",
    "2060",
    "2070",
    "2080",
    "2090",
    "2100"
]

# =============================================================================
# PATH-DEPENDENT FORECASTING
# =============================================================================

for i, yr in enumerate(years):

    pop_col = f"ssp2{yr}"

    # -------------------------------------------------------------------------
    # Create future prediction dataframe
    # -------------------------------------------------------------------------

    future_df = df_model.copy()

    future_df["log_pop"] = np.log(
        np.maximum(
            future_df[pop_col],
            1
        )
    )

    # -------------------------------------------------------------------------
    # Predict roadway demand implied by future population
    # -------------------------------------------------------------------------

    future_model_rl = np.exp(
        final_model.predict(future_df)
    )

    # -------------------------------------------------------------------------
    # Determine previous decade
    # -------------------------------------------------------------------------

    if yr == "2030":

        prev_model_rl = (
            df_model["Model_RL_2020"]
        )

        prev_actual_rl = (
            df_model["cl_total_length_2020"]
        )

        prev_pop = (
            df_model["CensusPop_20"]
        )

    else:

        prev_yr = years[i-1]

        prev_model_rl = (
            df_model[
                f"Model_RL_{prev_yr}"
            ]
        )

        prev_actual_rl = (
            df_model[
                f"Predicted_RL_{prev_yr}"
            ]
        )

        prev_pop = (
            df_model[
                f"ssp2{prev_yr}"
            ]
        )

    # -------------------------------------------------------------------------
    # Save model prediction
    # -------------------------------------------------------------------------

    df_model[
        f"Model_RL_{yr}"
    ] = future_model_rl

    # -------------------------------------------------------------------------
    # Calculate decade growth factor
    # -------------------------------------------------------------------------

    growth_factor = (
        future_model_rl
        /
        prev_model_rl
    )

    # -------------------------------------------------------------------------
    # Apply growth factor to previous roadway stock
    # -------------------------------------------------------------------------

    forecast_rl = (
        prev_actual_rl
        *
        growth_factor
    )

    # -------------------------------------------------------------------------
    # Hysteresis rule:
    #
    # If population declines,
    # roadway network remains unchanged.
    # -------------------------------------------------------------------------

    forecast_rl = np.where(
        df_model[pop_col] > prev_pop,
        forecast_rl,
        prev_actual_rl
    )

    # -------------------------------------------------------------------------
    # Store forecast
    # -------------------------------------------------------------------------

    df_model[
        f"Predicted_RL_{yr}"
    ] = forecast_rl

# =============================================================================
# FORECAST GROWTH PERCENTAGES
# =============================================================================

for yr in years:

    df_model[
        f"GrowthPct_{yr}"
    ] = (
        (
            df_model[
                f"Predicted_RL_{yr}"
            ]
            -
            df_model[
                "cl_total_length_2020"
            ]
        )
        /
        df_model[
            "cl_total_length_2020"
        ]
    ) * 100

# =============================================================================
# FORECAST OUTPUT
# =============================================================================

forecast_table = df_model[
    [
        'GEOID',
        'State',
        'city_type',
        'CensusPop_20',
        'ssp22050',
        'ssp22100',
        'cl_total_length_2020',
        'Predicted_RL_2030',
        'Predicted_RL_2050',
        'Predicted_RL_2100',
        'GrowthPct_2100'
    ]
].round(0)

print(forecast_table.head())

# =============================================================================
# SUMMARY OF 2100 GROWTH
# =============================================================================

print(
    df_model[
        'GrowthPct_2100'
    ].describe()
)

                            OLS Regression Results                            
Dep. Variable:                 log_rl   R-squared:                       0.895
Model:                            OLS   Adj. R-squared:                  0.894
Method:                 Least Squares   F-statistic:                     4134.
Date:                Sun, 28 Jun 2026   Prob (F-statistic):               0.00
Time:                        16:44:58   Log-Likelihood:                -14635.
No. Observations:               24425   AIC:                         2.937e+04
Df Residuals:                   24374   BIC:                         2.978e+04
Df Model:                          50                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.9038      0.046  

In [ ]:
df_model[['GEOID', 'State', 'REGION', 'city_type', 'ALAND', 'CensusPop_20',
       'ssp22030', 'ssp22040', 'ssp22050', 'ssp22060', 'ssp22070', 'ssp22080',
       'ssp22090', 'ssp22100', 'cl_total_length_2020','Predicted_RL_2020',
       'Predicted_RL_2030', 'Predicted_RL_2040', 'Predicted_RL_2050']]

,GEOID,State,REGION,city_type,ALAND,CensusPop_20,ssp22030,ssp22040,ssp22050,ssp22060,ssp22070,ssp22080,ssp22090,ssp22100,cl_total_length_2020,Predicted_RL_2020,Predicted_RL_2030,Predicted_RL_2040,Predicted_RL_2050
0,0102260,1.0,South,rural,5289895.0,1321.0,1469.730716,1594.447908,1703.018820,1807.843816,1907.902955,1990.668818,2042.540791,2076.679218,42480.438268,42480.438268,44540.112899,46179.454350,47549.376338
1,0151264,1.0,South,rural,646347.0,47.0,53.098290,58.771865,63.719030,68.636118,73.435686,77.437249,80.126641,82.589134,4847.498766,4847.498766,5117.173363,5352.981333,5548.455569
2,0171496,1.0,South,rural,8895669.0,796.0,806.213708,796.949969,777.721488,758.493050,736.833280,707.414625,672.961045,636.435207,42007.387675,42007.387675,42245.738816,42245.738816,42245.738816
3,0176872,1.0,South,suburban,12861624.0,2526.0,2577.677830,2585.723001,2571.506064,2569.510464,2564.707292,2538.345233,2497.361350,2446.128478,68898.738166,68898.738166,69520.742832,69616.950128,69616.950128
4,0130880,1.0,South,rural,6634644.0,269.0,280.068705,285.673821,289.707591,294.833775,300.161517,303.137626,302.647638,297.693833,18521.969725,18521.969725,18856.396609,19022.946017,19141.682871
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30573,5682967,56.0,West,rural,5957495.0,41.0,42.508509,43.558406,44.367530,45.676214,46.835851,47.323946,47.571269,47.658078,10043.795432,10043.795432,10206.141690,10317.248371,10401.862490
30574,5683100,56.0,West,rural,3001103.0,94.0,97.691612,99.530291,102.073548,106.254013,111.194528,114.257454,117.101573,118.089677,13739.065627,13739.065627,13975.949055,14092.076864,14250.754829
30575,5683765,56.0,West,rural,59631986.0,1567.0,1763.486088,1930.161922,2095.200636,2255.644610,2408.006355,2551.562793,2677.721579,2780.704590,114191.789931,114191.789931,120337.759711,125258.602182,129903.280765
30576,5684852,56.0,West,rural,43264734.0,118.0,135.082935,152.891701,172.274337,194.224381,217.969403,241.602383,264.367230,282.891668,37098.129351,37098.129351,39392.164406,41617.656380,43881.501018


In [ ]:
df_model.groupby('city_type')[['GrowthPct_2030', 'GrowthPct_2040', 'GrowthPct_2050', 'GrowthPct_2060',
       'GrowthPct_2070', 'GrowthPct_2080', 'GrowthPct_2090', 'GrowthPct_2100']].mean()

,GrowthPct_2030,GrowthPct_2040,GrowthPct_2050,GrowthPct_2060,GrowthPct_2070,GrowthPct_2080,GrowthPct_2090,GrowthPct_2100
city_type,,,,,,,,
not enough data,0.756466,1.322012,1.970552,2.991060,4.015212,5.059574,6.341641,7.596748
periurban,2.967444,5.237901,7.105869,8.859492,10.416002,11.572312,12.314486,12.802303
rural,1.193532,2.024656,2.743478,3.512679,4.225340,4.813547,5.280477,5.655860
suburban,2.871697,5.057075,6.854588,8.558907,10.064425,11.171026,11.869653,12.300756
urban,3.677580,6.625858,9.096297,11.379861,13.349294,14.748705,15.624251,16.108946


### Use mean growth pct for uban cities to forecast values for DC

In [ ]:
df_DC = roads_with_pop[roads_with_pop['State'] ==11][['GEOID', 'State', 'REGION', 'city type', 'ALAND', 'CensusPop_20',
       'ssp22030', 'ssp22040', 'ssp22050', 'ssp22060', 'ssp22070', 'ssp22080',
       'ssp22090', 'ssp22100', 'cl_total_length_2020']].reset_index(drop=True)

# -----------------------------------------------------------------------------
# Mean growth factors by city type
# -----------------------------------------------------------------------------
growth_cols = [
    'GrowthPct_2030', 'GrowthPct_2040', 'GrowthPct_2050', 'GrowthPct_2060',
    'GrowthPct_2070', 'GrowthPct_2080', 'GrowthPct_2090', 'GrowthPct_2100'
]

growth_lookup = (
    df_model
    .groupby('city_type')[growth_cols]
    .mean()
)

# -----------------------------------------------------------------------------
# New row (one-row dataframe)
# -----------------------------------------------------------------------------
new_row = df_DC.copy()          # dataframe containing the new observation

# Match column name
new_row = new_row.rename(columns={'city type':'city_type'})

# -----------------------------------------------------------------------------
# Add growth factors
# -----------------------------------------------------------------------------
new_row = new_row.merge(
    growth_lookup,
    left_on='city_type',
    right_index=True,
    how='left'
)

# -----------------------------------------------------------------------------
# Create future roadway length projections
# -----------------------------------------------------------------------------
years = [2030,2040,2050,2060,2070,2080,2090,2100]

for yr in years:
    growth_col = f'GrowthPct_{yr}'
    pred_col = f'Predicted_RL_{yr}'

    new_row[pred_col] = (
        new_row['cl_total_length_2020'] *
        (1 + new_row[growth_col] / 100)
    )

# -----------------------------------------------------------------------------
# 2020 prediction (actual value)
# -----------------------------------------------------------------------------
new_row['Predicted_RL_2020'] = new_row['cl_total_length_2020']

# -----------------------------------------------------------------------------
# Per-capita roadway length
# -----------------------------------------------------------------------------
new_row['length_m_perCap_2020'] = (
    new_row['cl_total_length_2020'] /
    new_row['CensusPop_20']
)

for yr in years:
    new_row[f'length_m_perCap_{yr}'] = (
        new_row[f'Predicted_RL_{yr}'] /
        new_row[f'ssp2{yr}']
    )

# -----------------------------------------------------------------------------
# Optional: create placeholder columns so the dataframe matches df_model
# -----------------------------------------------------------------------------
placeholder_cols = [
    'log_pop','log_area','log_rl',
    'Model_RL_2020','Model_RL_2030','Model_RL_2040',
    'Model_RL_2050','Model_RL_2060','Model_RL_2070',
    'Model_RL_2080','Model_RL_2090','Model_RL_2100'
]

for col in placeholder_cols:
    new_row[col] = np.nan

# -----------------------------------------------------------------------------
# Reorder columns to match df_model
# -----------------------------------------------------------------------------
new_row = new_row[df_model.columns]

# Append
df_model = pd.concat([df_model, new_row], ignore_index=True)

In [ ]:
# df_model[['GEOID', 'State', 'city_type','CensusPop_20', 'cl_total_length_2020','Model_RL_2020', 'Predicted_RL_2020', 'Model_RL_2030',
#        'Predicted_RL_2030', 'Model_RL_2040', 'Predicted_RL_2040',
#        'Model_RL_2050', 'Predicted_RL_2050', 'Model_RL_2060',
#        'Predicted_RL_2060', 'Model_RL_2070', 'Predicted_RL_2070',
#        'Model_RL_2080', 'Predicted_RL_2080', 'Model_RL_2090',
#        'Predicted_RL_2090', 'Model_RL_2100', 'Predicted_RL_2100','GrowthPct_2100']].sort_values('GrowthPct_2100')

In [ ]:
df_model[['CensusPop_20','cl_total_length_2020', 'GrowthPct_2050','GrowthPct_2080','GrowthPct_2100']].quantile([.05, .95])

,CensusPop_20,cl_total_length_2020,GrowthPct_2050,GrowthPct_2080,GrowthPct_2100
0.05,78.0,4902.786596,0.000000,0.00000,0.000000
0.95,29575.0,377345.046537,16.632545,29.07468,34.167156


In [ ]:
df_model['length_m_perCap_2020'] = df_model['cl_total_length_2020']/df_model['CensusPop_20']

# Years to process
years = range(2030, 2110, 10)

for year in years:
    df_model[f'length_m_perCap_{year}'] = (
        df_model[f'Predicted_RL_{year}'] / df_model[f'ssp22{str(year)[1:]}']
    )

In [ ]:
df_model_selected = df_model[['GEOID', 'length_m_perCap_2020','length_m_perCap_2030', 'length_m_perCap_2040', 'length_m_perCap_2050', 'length_m_perCap_2060', 'length_m_perCap_2070', 'length_m_perCap_2080',
                               'length_m_perCap_2090', 'length_m_perCap_2100']]
roads_attr_selected = roads_with_pop_all[['GEOID','label','future trend from SSP 2', 'weighted_HU_density_sqmi', 'median_income', 'REGION', 'NAMELSAD','city type']]

df_output = df_model_selected.merge(roads_attr_selected, on ='GEOID', how='left')

df_output.to_csv(r"/content/drive/MyDrive/quantify-infrastructure-code/outputfiles/csvs/RE_roads_perCap_ssp2_local.csv")

In [ ]:
30.435215-29.959, 34.1-30.4

(0.47621499999999983, 3.700000000000003)

In [ ]:
---

SyntaxError: invalid syntax (1947214667.py, line 1)

### Check data transformation stats

In [ ]:
import numpy as np
import pandas as pd

df = roads_with_pop_all[['GEOID', 'REGION', 'city type',
                         'CensusPop_20', 'cl_total_length_2020']].copy()

df = df.dropna()

# Remove zeros for log/Box-Cox
df = df[(df['CensusPop_20'] > 0) &
        (df['cl_total_length_2020'] > 0)]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.scatter(df['CensusPop_20'],
            df['cl_total_length_2020'],
            alpha=0.6)

plt.xlabel('Population')
plt.ylabel('Centerline Length')
plt.title('Raw Scale')
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(np.log10(df['CensusPop_20']),
            np.log10(df['cl_total_length_2020']),
            alpha=0.6)

plt.xlabel('log10(Population)')
plt.ylabel('log10(Centerline Length)')
plt.title('Log-Log Relationship')
plt.show()

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(np.log(df['CensusPop_20']))
y = np.log(df['cl_total_length_2020'])

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
df['sqrt_pop'] = np.sqrt(df['CensusPop_20'])

plt.figure(figsize=(8,6))
plt.scatter(df['sqrt_pop'],
            df['cl_total_length_2020'],
            alpha=0.6)

plt.xlabel('sqrt(Population)')
plt.ylabel('Centerline Length')
plt.title('Centerline Length vs sqrt(Population)')
plt.show()

In [ ]:
X = sm.add_constant(df['sqrt_pop'])
y = df['cl_total_length_2020']

sqrt_model = sm.OLS(y, X).fit()

print(sqrt_model.summary())

In [ ]:
print("Log-log R²:", model.rsquared)
print("Sqrt-pop R²:", sqrt_model.rsquared)

In [ ]:
from scipy.stats import boxcox

y_bc, lambda_y = boxcox(df['cl_total_length_2020'])
x_bc, lambda_x = boxcox(df['CensusPop_20'])

print("Population lambda:", lambda_x)
print("Length lambda:", lambda_y)

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(x_bc,
            y_bc,
            alpha=0.6)

plt.xlabel(f'Population Box-Cox (λ={lambda_x:.2f})')
plt.ylabel(f'Length Box-Cox (λ={lambda_y:.2f})')
plt.title('Box-Cox Transformed Relationship')

plt.show()

In [ ]:
X = sm.add_constant(x_bc)

bc_model = sm.OLS(y_bc, X).fit()

print(bc_model.summary())

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Log-Log', 'Sqrt Population', 'Box-Cox'],
    'R2': [
        model.rsquared,
        sqrt_model.rsquared,
        bc_model.rsquared
    ]
})

print(comparison.sort_values('R2', ascending=False))

In [ ]:
import scipy.stats as stats

resid = model.resid

fig, ax = plt.subplots(1,2, figsize=(12,5))

ax[0].hist(resid, bins=30)

stats.probplot(resid, dist='norm', plot=ax[1])

plt.show()

### State level plots

In [ ]:
df_plot = df_forecast[
    df_forecast['ssp22030'] > df_forecast['CensusPop_20']
][[
    'GEOID',
    'State',
    'city_type',
    'cl_total_length_2020',
    'Predicted_RL_2020',
    'percent_error'
]].copy()

In [ ]:
plt.figure(figsize=(8,8))

sns.scatterplot(
    data=df_plot,
    x='cl_total_length_2020',
    y='Predicted_RL_2020',
    hue='city_type',
    alpha=0.7
)

xmin = min(
    df_plot['cl_total_length_2020'].min(),
    df_plot['Predicted_RL_2020'].min()
)

xmax = max(
    df_plot['cl_total_length_2020'].max(),
    df_plot['Predicted_RL_2020'].max()
)

plt.plot(
    [xmin, xmax],
    [xmin, xmax],
    'k--',
    linewidth=2,
    label='1:1 line'
)

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Observed Roadway Length (2020)')
plt.ylabel('Predicted Roadway Length (2020)')
plt.title('Observed vs Predicted Roadway Length')
plt.legend(bbox_to_anchor=(1.05,1))
plt.tight_layout()
plt.show()

In [ ]:
g = sns.FacetGrid(
    df_plot,
    col='State',
    hue='city_type',
    col_wrap=4,
    height=4,
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x='cl_total_length_2020',
    y='Predicted_RL_2020',
    alpha=0.7
)

for ax in g.axes.flat:

    ax.set_xscale('log')
    ax.set_yscale('log')

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    low = min(xmin, ymin)
    high = max(xmax, ymax)

    ax.plot(
        [low, high],
        [low, high],
        'k--',
        linewidth=1
    )

g.add_legend()

g.set_axis_labels(
    'Observed RL',
    'Predicted RL'
)

plt.show()

In [ ]:
state_perf = (
    df_plot
    .groupby('State')
    .agg(
        n=('GEOID','count'),
        mean_error=('percent_error','mean'),
        median_abs_error=('percent_error',
                          lambda x: np.median(np.abs(x))),
        rmse_pct=('percent_error',
                  lambda x: np.sqrt(np.mean(x**2)))
    )
    .sort_values('rmse_pct')
)

state_perf

In [ ]:
df_forecasts[df_forecasts['ssp22030'] > df_forecasts['CensusPop_20']]['percent_error'].describe().round(2)

In [ ]:
----

### Check stats

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(
    np.log(df_forecasts['percent_error']),
    bins=30,
    kde=True
)

plt.xlabel('Road Length')
plt.ylabel('Count')
plt.show()

In [ ]:
# # Select relevant columns
# df = roads_with_pop_all[
#     ['REGION', 'city type', 'CensusPop_20', 'cl_total_length_2020']
# ].copy()
# # Remove missing or non-positive values (required for log scale)
# df = df.dropna()
# df = df[
#     (df['CensusPop_20'] > 0) &
#     (df['cl_total_length_2020'] > 0)
# ]

# # Faceted scatter plot
# g = sns.FacetGrid(
#     df,
#     col='city type',
#     hue='REGION', #
#     col_wrap=3,      # adjust depending on number of regions
#     height=4,
#     sharex=False,
#     sharey=False
# )

# g.map_dataframe(
#     sns.scatterplot,
#     x='CensusPop_20',
#     y='cl_total_length_2020',
#     alpha=0.7
# )

# # Set log scales for each subplot
# for ax in g.axes.flat:
#     ax.set_xscale('log')
#     ax.set_yscale('log')
#     ax.set_xlabel('Population (2020)')
#     ax.set_ylabel('Road Length (2020)')

# g.add_legend()
# plt.tight_layout()
# plt.show()

In [ ]:
----

### Create Maps

In [ ]:
import os
import geopandas as gpd
os.environ['USE_PYGEOS'] = '0'
# Import places map
places_US = gpd.read_file(r'Maps_2020\all_places\compiled.shp')
print('Total no of places in the US:===')
print(places_US.shape)

# Exclude states that do not have consistent data for roadways
places_50_states = places_US[(places_US['STATEFP'] != '02') & (places_US['STATEFP'] != '60') & (places_US['STATEFP'] != '66') & (places_US['STATEFP'] != '69') & (places_US['STATEFP'] != '72') & (places_US['STATEFP'] != '78')]
print('No of places in the 50 states:===')
print(places_50_states.shape)

In [ ]:
gdf = places_50_states.merge(df_forecasts, on ='GEOID', how = 'right')

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
fig, ax = plt.subplots(figsize=(12, 8))
# Set black backgrounds
fig.patch.set_facecolor("black")
ax.set_facecolor("black")
# Remove grid lines
ax.grid(False)
gdf.plot(ax=ax, column = 'city_type', legend= True)

In [ ]:
gdf[(gdf['percent_error'] < -30)].shape, gdf[(gdf['percent_error'] > 30)].shape, gdf[(gdf['percent_error'] >= -30) & (gdf['percent_error'] <= 30)].shape

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
fig, ax = plt.subplots(figsize=(12, 8))
# Set black backgrounds
# fig.patch.set_facecolor("black")
# ax.set_facecolor("black")
# Remove grid lines
ax.grid(False)

gdf[(gdf['percent_error'] < -30)].plot(ax=ax, column = 'percent_error', legend=True)
# (gdf['percent_error'] < -30) &

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
fig, ax = plt.subplots(figsize=(12, 8))
# Set black backgrounds
# fig.patch.set_facecolor("black")
# ax.set_facecolor("black")
# Remove grid lines
ax.grid(False)

gdf[(gdf['percent_error'] > -30) & (gdf['percent_error'] < 30)].plot(ax=ax, column = 'percent_error', legend=True)

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
fig, ax = plt.subplots(figsize=(12, 8))
# Set black backgrounds
fig.patch.set_facecolor("black")
ax.set_facecolor("black")
# Remove grid lines
ax.grid(False)


colors = {
    "Increasing": "red",
    "Decreasing": "green"
}

gdf_not_HI = gdf[gdf['STATEFP'] != '15']

for category, color in colors.items():
    gdf_not_HI[gdf_not_HI["city_type"] == category].plot(
    ax=ax,
    color=color,
    edgecolor = 'gray',
    linewidth=0.05,
    label=category
)


In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
fig, ax = plt.subplots(figsize=(12, 8))
# Set black backgrounds
fig.patch.set_facecolor("black")
ax.set_facecolor("black")
# Remove grid lines
ax.grid(False)


colors = {
    "Increasing": "red",
    "Decreasing": "green"
}

gdf_not_HI = gdf[gdf['STATEFP'] != '15']

for category, color in colors.items():
    gdf_not_HI[gdf_not_HI["trend_2050_to_2100"] == category].plot(
    ax=ax,
    color=color,
    edgecolor = 'gray',
    linewidth=0.05,
    label=category
)

In [ ]:
---

In [ ]:
# # # Import cartographic base maps: Cartographic boundary for plotting
# US_counties_cb = gpd.read_file(r'Maps_2020\cb_2020_us_county_5m.zip') # tl_2020_us_county
# US_states_cb = gpd.read_file(r'Maps_2020\cb_2020_us_state_5m.zip')  # tl_2020_us_state
# # source: https://gis.stackexchange.com/questions/141580/which-projection-is-best-for-mapping-the-contiguous-united-states
# US_states_cb = US_states_cb.to_crs('EPSG:9311')  # FOR PLOTLY MAPBOX DO NOT CHANGE PROJECTION, SO COMMENT THESE LINES
# US_counties_cb = US_counties_cb.to_crs('EPSG:9311')